# TPU v5e-8: 32K chat SFT from the mid-trained checkpoint (MTP ablation)

Select **TPU v5e-8**, enable Internet, and attach:

1. the **chat SFT corpus** saved by `nano_dsv41f_prepare_chat_sft_cpu.ipynb`;
2. `xiayicheng3gmailcom/nanodsv4-1f-sftcheckpoint`, which holds the completed
   **mid-training checkpoint** (`midtrain/checkpoints`, step 85,085) used as the start;
3. `xiayicheng3gmailcom/nanodsv4-1fposttraincorpus` for the optional reasoning slice.

This replaces the earlier reasoning/agent SFT with chat data and keeps that stage's
recipe: fresh optimizer, 2.6e-5 → 2.6e-6 cosine LR, YaRN ×4 to 32K and the
candidate-masked indexer auxiliary. `REASONING_FRACTION` keeps a small share of the
existing reasoning traces so thinking mode keeps its format; agent traces are dropped.

**Ablation:** run this notebook twice with identical settings except `MTP_WEIGHT`:
`0.0` (baseline) and `0.1` (DSpark multi-token-prediction loss with gradients into
the backbone, 256 sampled anchors per 32K row). Compare held-out chat loss
(logged at step 0 and every 500 updates), benchmarks and samples.

A one-pass run is about 3,540 updates (441M tokens); the previous 32K SFT measured ~2.6 s/update,
so a run should finish in one session. Checkpoints are saved before compilation and
every 500 updates; set `RESUME_CHECKPOINT` to continue a paused run.

In [ ]:
import os
import time
SESSION_STARTED = time.time()
SESSION_HOURS = 8.0
MTP_WEIGHT = 0.0  # 0.0 = baseline arm, 0.1 = MTP arm; keep unchanged on resume
MTP_ANCHORS = 256
REASONING_FRACTION = 0.05  # 28M-token reasoning pool: 5% keeps ~98% of the chat corpus in one pass
SFT_TOKENS = 0  # 0 = one pass over the chat corpus at this mix
MAX_STEPS = 0  # >0 only for a smoke test
RESUME_CHECKPOINT = ''  # later sessions: this run's training/sft/checkpoints directory
os.environ['NANO_DSV41F_REF'] = 'main'

In [ ]:
"""Canonical notebook bootstrap. Standard library only: runs before importing JAX."""
from pathlib import Path
from importlib.metadata import version
import os
import subprocess
import sys

if any(name in sys.modules for name in ("jax", "jaxlib", "libtpu", "tokamax", "nano_dsv41f")):
    raise RuntimeError("Restart the Kaggle session and run this cell first: JAX/TPU was already imported.")

ROOT = Path('/kaggle/working/nano-dsv4.1f')
REPO_REF = os.environ.get('NANO_DSV41F_REF', 'main')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(ROOT)], check=True)
if not (ROOT / '.git').exists():
    raise RuntimeError(f'{ROOT} exists but is not a git checkout.')
status = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain',
                                  '--untracked-files=no'], text=True)
if status.strip():
    raise RuntimeError('Checkout has tracked edits; preserve them before updating:\n' + status)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
# Pin the entire validated package combination, not just the Python jax package.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                '-r', str(ROOT / 'requirements-tpu.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT), '--no-deps'], check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['JAX_PLATFORMS'] = 'tpu'
print('repo:', ROOT)
print('requested ref:', REPO_REF)
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('packages:', {p: version(p) for p in ('jax', 'jaxlib', 'libtpu', 'tokamax')})
# Execution cells run fresh Python processes. Never retain an old TPU client in IPython.


In [ ]:
import json
from datetime import datetime, timezone
def find_one(pattern, keep=lambda p: True, what='input'):
    hits = sorted({p.parent for p in Path('/kaggle/input').rglob(pattern) if keep(p)})
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one {what}, found {len(hits)}: {hits[:4]}')
    return hits[0]
chat_corpus = find_one('chat_sft_manifest.json', what='chat SFT corpus')
midtrain = find_one('latest.json', lambda p: p.parent.name == 'checkpoints' and p.parent.parent.name == 'midtrain',
                    'midtrain checkpoint directory')
reasoning = find_one('posttrain_manifest.json', what='v3 posttrain corpus') if REASONING_FRACTION else None
arm = 'mtp' if MTP_WEIGHT else 'baseline'
RUN_ROOT = Path('/kaggle/working') / (f'chat-sft-{arm}-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
RUN_ROOT.mkdir(exist_ok=False)
OUTPUT = RUN_ROOT / 'training'
subprocess.run(['git', 'archive', '--format=tar.gz', '-o', str(RUN_ROOT / 'source.tar.gz'), 'HEAD'], check=True)
command = [sys.executable, '-u', 'scripts/run_chat_sft.py', '--chat-corpus', str(chat_corpus),
           '--output', str(OUTPUT), '--mtp-weight', str(MTP_WEIGHT), '--mtp-anchors', str(MTP_ANCHORS),
           '--reasoning-fraction', str(REASONING_FRACTION), '--sft-tokens', str(SFT_TOKENS),
           '--max-steps', str(MAX_STEPS),
           '--deadline-unix', str(SESSION_STARTED + SESSION_HOURS * 3600)]
command += ['--resume', RESUME_CHECKPOINT] if RESUME_CHECKPOINT else ['--midtrain', str(midtrain)]
if reasoning:
    command += ['--reasoning-corpus', str(reasoning)]
print(' '.join(command))
with (RUN_ROOT / 'train.log').open('w', buffering=1) as log:
    p = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in p.stdout:
            if 'Autotuning cache miss' in line:
                continue  # Tokamax autotune notices; kept out of the log for readability
            print(line, end='', flush=True); log.write(line)
        rc = p.wait()
    except KeyboardInterrupt:
        p.terminate()
        print('Waiting for the runner to checkpoint between updates')
        rc = p.wait()
summary = json.loads((OUTPUT / 'summary.json').read_text()) if (OUTPUT / 'summary.json').exists() else {}
print(json.dumps({k: v for k, v in summary.items() if k != 'error'}, indent=2))
if rc:
    print(summary.get('error', ''))
    raise RuntimeError('Training failed; inspect train.log. The previous committed checkpoint is retained.')

## Outputs

`training/summary.json` holds the plan, held-out losses and the final checkpoint;
`metrics.jsonl` the training curve (including MTP loss and per-depth draft accuracy);
`sft-compile-*.json` the compiler memory/collective diagnostics. Save this version's
output; the final `sft/checkpoints` directory exports with
`scripts/export_posttrain_checkpoint.py` like the previous SFT.